# Training analysis — rewriter + drift coefficients

Fill in W&B run paths or local artifact paths below. This notebook is a **template**: it expects you to have logged training with `wandb` (optional) and saved `drift_coef_opt` outputs under `outputs/cotrain/`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
import pandas as pd

ROOT = Path("..").resolve()
TRAJ = ROOT / "outputs/cotrain/drift_coef_opt_trajectory.jsonl"
OPT_JSON = ROOT / "outputs/cotrain/optimized_drift_coefs.json"

# W&B: set your entity/project and run ids for r=4,8,16 LoRA ablations
WANDB_RUNS = {
    "r4": None,
    "r8": None,
    "r16": None,
}

## Section 1 — Rewriter training diagnostics

In [ ]:
# Cell 1.1 — Training / validation loss (W&B API or exported CSV)
try:
    import wandb

    api = wandb.Api()
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, run_id in WANDB_RUNS.items():
        if not run_id:
            continue
        run = api.run(run_id)
        hist = run.history(keys=["train/loss", "eval/loss"], pandas=True)
        if "train/loss" in hist:
            hist["train/loss"].dropna().plot(ax=ax, label=f"{label} train")
        if "eval/loss" in hist:
            hist["eval/loss"].dropna().plot(ax=ax, label=f"{label} val", linestyle="--")
    ax.set_title("Rewriter loss by LoRA rank")
    ax.set_xlabel("step")
    ax.legend()
    plt.show()
except Exception as e:
    print("W&B not configured or run ids missing:", e)

In [ ]:
# Cell 1.2 — Rewriter slop score vs step (logged as eval/rewriter_slop_mean)
try:
    import wandb

    api = wandb.Api()
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, run_id in WANDB_RUNS.items():
        if not run_id:
            continue
        run = api.run(run_id)
        hist = run.history(keys=["eval/rewriter_slop_mean"], pandas=True)
        if "eval/rewriter_slop_mean" in hist:
            hist["eval/rewriter_slop_mean"].dropna().plot(ax=ax, label=label)
    ax.set_title("Rewriter slop (lower is better)")
    ax.set_xlabel("step")
    ax.legend()
    plt.show()
except Exception as e:
    print("W&B rewriter slop unavailable:", e)

In [ ]:
# Cell 1.3 — Example prompts vs rewriter outputs (fill after running inference locally)
examples = [
    # {"topic": "...", "input": "...", "rewritten": "...", "slop": 0.0},
]
pd.DataFrame(examples)

## Section 2 — Drift coefficient optimization diagnostics

In [ ]:
# Cell 2.1 — Adam trajectory: coefficients + approximate fool rate
rows = []
if TRAJ.is_file():
    for line in TRAJ.read_text().splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
df = pd.DataFrame(rows)
if len(df):
    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    axes[0].plot(df["step"], df["alpha_semantic"], label="alpha_semantic")
    axes[0].plot(df["step"], df["alpha_rouge"], label="alpha_rouge")
    axes[0].plot(df["step"], df["alpha_bertscore"], label="alpha_bertscore")
    axes[0].set_ylabel("coefficient")
    axes[0].legend()
    axes[0].set_title("Drift penalty coefficients (Adam)")
    axes[1].plot(df["step"], df["fool_rate_approx"], label="approx")
    axes[1].plot(df["step"], df["fool_rate_hard"], label="hard", alpha=0.7)
    axes[1].set_ylabel("fool rate")
    axes[1].set_xlabel("step")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("No trajectory at", TRAJ)

In [ ]:
# Cell 2.2 — Grid vs Adam summary
if OPT_JSON.is_file():
    doc = json.loads(OPT_JSON.read_text())
    pd.DataFrame(
        [
            {"method": "grid", **doc.get("grid_search", {})},
            {"method": "adam", **doc.get("adam", {})},
        ]
    )
else:
    print("Run deslop/drift_coef_opt.py first; expected", OPT_JSON)

In [ ]:
# Cell 2.3 — Sensitivity: sweep each coefficient with the other two fixed at Adam optimum
if OPT_JSON.is_file():
    doc = json.loads(OPT_JSON.read_text())
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, key, xcol in zip(
        axes,
        (
            "sensitivity_alpha_semantic",
            "sensitivity_alpha_rouge",
            "sensitivity_alpha_bertscore",
        ),
        ("alpha_semantic", "alpha_rouge", "alpha_bertscore"),
    ):
        sens = doc.get(key, [])
        if sens:
            sdf = pd.DataFrame(sens)
            ax.plot(sdf[xcol], sdf["fool_rate_hard"], marker="o")
            ax.set_xlabel(xcol)
            ax.set_ylabel("fool_rate_hard")
    fig.suptitle("Sensitivity (two coefficients fixed at Adam optimum)")
    plt.tight_layout()
    plt.show()
else:
    print("Missing", OPT_JSON)

## Section 3 — End-to-end comparison

In [ ]:
# Cell 3.1 — Slop distributions (baseline vs evolutionary best vs rewriter)
# Replace with your measured scores on a fixed topic list.
slop_baseline = []
slop_evo = []
slop_rewriter = []
data = []
for s in slop_baseline:
    data.append({"cond": "generic", "slop": s})
for s in slop_evo:
    data.append({"cond": "evolutionary", "slop": s})
for s in slop_rewriter:
    data.append({"cond": "rewriter", "slop": s})
if data:
    df3 = pd.DataFrame(data)
    fig, ax = plt.subplots(figsize=(6, 4))
    for c in df3["cond"].unique():
        ax.hist(df3.loc[df3["cond"] == c, "slop"], alpha=0.45, label=c, bins=10)
    ax.set_xlabel("slop score")
    ax.legend()
    plt.show()
else:
    print("Populate slop_* lists with experiment results.")

In [ ]:
# Cell 3.2 — Amortization ratio (evolutionary Groq essay budget vs one rewriter forward per test topic)
# Numerator: co-training deslop essay Groq calls (same formula as loop estimate; mutator calls excluded).
# Denominator: len(test split IDs) × 1 forward pass per topic.
cotrain_path = ROOT / "configs" / "cotrain.yaml"
rewriter_path = ROOT / "configs" / "rewriter.yaml"
ct = yaml.safe_load(cotrain_path.read_text(encoding="utf-8"))
rw = yaml.safe_load(rewriter_path.read_text(encoding="utf-8"))
manifest_rel = rw.get("split_manifest_path", "outputs/cotrain/splits/rewriter_split_manifest.json")
manifest_path = Path(manifest_rel)
if not manifest_path.is_absolute():
    manifest_path = (ROOT / manifest_path).resolve()
n_test_topics = 0
if manifest_path.is_file():
    man = json.loads(manifest_path.read_text(encoding="utf-8"))
    n_test_topics = len(man.get("splits", {}).get("test", []))
num_rounds = int(ct.get("num_rounds", 0))
topics_per_round = int(ct.get("topics_per_round", 0))
population_size = int(ct.get("population_size", 0))
generations_per_topic = int(ct.get("generations_per_topic", 0))
essays_per_candidate = int(ct.get("essays_per_candidate", 0))
total_groq_essay_calls = (
    num_rounds * topics_per_round * population_size * generations_per_topic * essays_per_candidate
)
rewriter_inference_calls = n_test_topics * 1
if rewriter_inference_calls > 0:
    amortization_ratio = total_groq_essay_calls / rewriter_inference_calls
    summary = {
        "num_rounds": num_rounds,
        "topics_per_round": topics_per_round,
        "population_size": population_size,
        "generations_per_topic": generations_per_topic,
        "essays_per_candidate": essays_per_candidate,
        "total_groq_essay_calls_search": total_groq_essay_calls,
        "n_test_topics_manifest": n_test_topics,
        "rewriter_inference_calls": rewriter_inference_calls,
        "amortization_ratio": amortization_ratio,
    }
    print(json.dumps(summary, indent=2))
    print(f"\nAmortization ratio ≈ {amortization_ratio:,.1f}x Groq essay calls per rewriter forward (test topics).")
else:
    print(
        "Cannot compute ratio: test split empty or manifest missing. Build the split manifest first."
    )